# 平均 IQ 与类型化读取

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import sys
from pathlib import Path

import scopecat as sc

project = sc.open_project()
if Path(sys.prefix).resolve() != (project.root / ".venv").resolve():
    raise RuntimeError(f"请在 Select Kernel 中选择 {project.root / '.venv'}")

In [ ]:
import numpy as np

from lab_teaching.session import open_parameters

session = project.authoring()
session.refresh()
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

`src/my_experiment/teaching.py` 已包含逐 shot 和平均 IQ 两个实验。平均函数是普通 NumPy 代码，`@sc.compute` 让实验定义中的调用加入计算图。

`result_types.py` 中的 `type IQ = Annotated[complex, sc.Unit("ratio")]` 同时用于计算返回值和下面的读取字段。先运行，不需要手工拼装实验。

In [ ]:
from dataclasses import dataclass

from my_experiment.result_types import IQ
from my_experiment.teaching import mean_rabi


@dataclass(frozen=True)
class MeanRow:
    amplitude: sc.Quantity
    iq: IQ


scan = np.linspace(0, 0.8, 7)
raw = (
    session.prepare(teaching_rabi().sweep(amplitude=scan), parameters=params)
    .run()
    .wait(timeout=120)
    .result()
)
mean = (
    session.prepare(mean_rabi().sweep(amplitude=scan), parameters=params)
    .run()
    .wait(timeout=120)
    .result()
)
rows = mean.result().rows_as(MeanRow)
np.testing.assert_allclose(
    [row.iq for row in rows],
    np.asarray(raw.measurements()["iq"].require_values()).mean(axis=1),
)
assert len(rows) == 7
print(rows[0])

保存身份后可以重启 kernel；只运行检查内核、下面的导入与读取单元，即可读取既有记录，无需重新采集。

In [ ]:
import json

(project.root / "mean-run.json").write_text(
    json.dumps({"run": mean.id}), encoding="utf-8"
)

In [ ]:
import json
from dataclasses import dataclass


@dataclass(frozen=True)
class MeanRow:
    amplitude: sc.Quantity
    iq: IQ


with project.authoring() as reader:
    run_id = json.loads((project.root / "mean-run.json").read_text(encoding="utf-8"))[
        "run"
    ]
    reopened = reader.run(run_id).result().rows_as(MeanRow)
    assert len(reopened) == 7
    print("重开:", reopened[0])

小修改：在 `teaching.py` 的 `mean_iq` 中改变一个计算因子；保存后重跑 refresh/import 并创建新请求。对照断言会指出科学含义发生变化，先解释差异再修改预期。
平均结果没有 shot 分布，不应直接传给依赖 shot 误差估计的拟合函数。